# 02 Data checks (July)

Read-only counts that decide what the late label means: final stops (3a), duplicates (3b), extreme delays (3c) and planned hours per service and station (6). Nothing here changes the saved outputs of notebook 01.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA_DIR = Path("../data")
MONTHS = ["2026-04", "2026-05", "2026-06", "2026-07", "2026-08", "2026-09"]  # files in data/
CHECK_MONTH = "2026-07"   # the month these checks run on

DELAY_COLUMN = "delay_in_min"
RIDE_KEY = ["train_line_ride_id", "train_line_station_num"]        # one stop of one ride
TIMETABLE_KEY = ["eva", "train_number", "departure_planned_time"]  # same train, station, planned time
SERVICE_KEY = ["train_type", "train_number"]                       # a recurring service, e.g. ICE 123
CANCEL_COLUMNS = ["arrival_is_canceled", "departure_is_canceled"]
EXTREME_DELAY_MIN = 180   # PROPOSAL, not a decision: |delay| above this counts as extreme
EXAMPLE_ROWS = 6          # how many example rows to show

CHECK_COLUMNS = [
    "eva", "train_type", "train_number", DELAY_COLUMN,
    *RIDE_KEY,
    "arrival_planned_time", "arrival_change_time",
    "departure_planned_time", "departure_change_time",
    *CANCEL_COLUMNS,
]

In [3]:
def month_path(month: str) -> Path:
    """Path of the Parquet file for one month (temporary copy from notebook 01; moves to src/)."""
    if month not in MONTHS:
        raise ValueError(f"Unknown month {month!r}; expected one of {MONTHS}")
    return DATA_DIR / f"data-{month}.parquet"


df = pd.read_parquet(month_path(CHECK_MONTH), columns=CHECK_COLUMNS)
print("Month:", CHECK_MONTH)
print("Rows and columns:", df.shape)   # July must show 14,768,104 rows and 12 columns

Month: 2026-07
Rows and columns: (14768104, 12)


In [4]:
def share(part: int, whole: int) -> str:
    """Format a count as 'part of whole (x.x%)'."""
    percent = f"{part / whole:.1%}" if whole else "n/a"
    return f"{part:,} of {whole:,} ({percent})"


def minutes_between(later: pd.Series, earlier: pd.Series) -> pd.Series:
    """Difference between two time columns, in minutes."""
    return (later - earlier).dt.total_seconds() / 60


def delay_match(stops: pd.DataFrame, rows: pd.Series, change_col: str, planned_col: str) -> None:
    """Print how often delay_in_min equals change time minus planned time on the chosen rows.

    Rows missing either time are counted separately, never treated as a mismatch.
    """
    subset = stops.loc[rows, [DELAY_COLUMN, change_col, planned_col]]
    has_times = subset[change_col].notna() & subset[planned_col].notna()
    with_times = subset[has_times]
    computed = minutes_between(with_times[change_col], with_times[planned_col])
    matches = int((computed == with_times[DELAY_COLUMN]).sum())
    print("  Missing a time (not compared):", share(int((~has_times).sum()), len(subset)))
    print(f"  Delay = {change_col} - {planned_col}:", share(matches, len(with_times)))

## 3a Final stops

A ride's last stop has no departure. Which delay does `delay_in_min` hold there? We compare it with arrival change minus arrival planned. As a control, the same comparison for departures on normal rows.

In [5]:
no_departure = df["departure_planned_time"].isna()
has_delay = df[DELAY_COLUMN].notna()
last_stop_num = df.groupby("train_line_ride_id")["train_line_station_num"].transform("max")
is_last_stop = df["train_line_station_num"] == last_stop_num

n_no_departure = int(no_departure.sum())
print("Rows with no planned departure:", share(n_no_departure, len(df)))
print("  ...that are the last stop of their ride:", share(int((no_departure & is_last_stop).sum()), n_no_departure))
print("  ...that have a delay value:             ", share(int((no_departure & has_delay).sum()), n_no_departure))
print()
print(pd.crosstab(no_departure.rename("no planned departure"),
                  has_delay.rename("delay present"), margins=True))

print("\nRows with no planned departure but a delay: is it the ARRIVAL delay?")
delay_match(df, no_departure & has_delay, "arrival_change_time", "arrival_planned_time")

print("\nRows with a planned departure and a delay: is it the DEPARTURE delay?")
delay_match(df, ~no_departure & has_delay, "departure_change_time", "departure_planned_time")

Rows with no planned departure: 1,618,335 of 14,768,104 (11.0%)
  ...that are the last stop of their ride: 964,047 of 1,618,335 (59.6%)
  ...that have a delay value:              1,126,514 of 1,618,335 (69.6%)

delay present          False      True       All
no planned departure                            
False                      0  13149769  13149769
True                  491821   1126514   1618335
All                   491821  14276283  14768104

Rows with no planned departure but a delay: is it the ARRIVAL delay?
  Missing a time (not compared): 0 of 1,126,514 (0.0%)
  Delay = arrival_change_time - arrival_planned_time: 1,126,514 of 1,126,514 (100.0%)

Rows with a planned departure and a delay: is it the DEPARTURE delay?
  Missing a time (not compared): 0 of 13,149,769 (0.0%)
  Delay = departure_change_time - departure_planned_time: 13,149,769 of 13,149,769 (100.0%)


## 3b Duplicates by key

"Duplicate rows: 0" in notebook 01 only finds rows that are identical in every column. Here we look for the same stop recorded twice: by ride key (ride id + stop number) and by timetable key (station + train number + planned departure).

In [6]:
# duplicated() marks EXTRA copies: a group of 3 identical keys counts as 2.
ride_dupes = df.duplicated(subset=RIDE_KEY)
print("Extra copies by ride key:     ", share(int(ride_dupes.sum()), len(df)))

# Rows without a planned departure all share NaT, so the same train at the same
# station on different days would look like duplicates. Only rows with a departure are checked.
has_departure = df["departure_planned_time"].notna()
timetable_dupes = df.loc[has_departure, TIMETABLE_KEY].duplicated()
print("Extra copies by timetable key:", share(int(timetable_dupes.sum()), int(has_departure.sum())))

in_ride_group = df.duplicated(subset=RIDE_KEY, keep=False)   # every row of a duplicate group
print("\nExample duplicate rows (ride key):")
display(df[in_ride_group].sort_values(RIDE_KEY).head(EXAMPLE_ROWS))

in_timetable_group = has_departure & df.duplicated(subset=TIMETABLE_KEY, keep=False)
print("Example duplicate rows (timetable key):")
display(df[in_timetable_group].sort_values(TIMETABLE_KEY).head(EXAMPLE_ROWS))

Extra copies by ride key:      13,676,228 of 14,768,104 (92.6%)
Extra copies by timetable key: 1,255 of 13,149,769 (0.0%)

Example duplicate rows (ride key):


,eva,train_type,train_number,delay_in_min,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,arrival_is_canceled,departure_is_canceled
65743,08010133,IC,2456,0.0,-1000672545375930240,1,NaT,NaT,2026-07-01 06:38:00,2026-07-01 06:38:00,False,True
597051,08010133,IC,2456,0.0,-1000672545375930240,1,NaT,NaT,2026-07-02 06:38:00,2026-07-02 06:38:00,False,True
70235,08011956,IC,2456,0.0,-1000672545375930240,2,2026-07-01 06:46:00,2026-07-01 06:46:00,2026-07-01 06:47:00,2026-07-01 06:47:00,True,True
601760,08011956,IC,2456,0.0,-1000672545375930240,2,2026-07-02 06:46:00,2026-07-02 06:46:00,2026-07-02 06:47:00,2026-07-02 06:47:00,True,True
79824,08010366,IC,2456,1.0,-1000672545375930240,3,2026-07-01 07:02:00,2026-07-01 07:02:00,2026-07-01 07:04:00,2026-07-01 07:05:00,True,False
610361,08010366,IC,2456,0.0,-1000672545375930240,3,2026-07-02 07:02:00,2026-07-02 07:02:00,2026-07-02 07:04:00,2026-07-02 07:04:00,True,True


Example duplicate rows (timetable key):


,eva,train_type,train_number,delay_in_min,train_line_ride_id,train_line_station_num,arrival_planned_time,arrival_change_time,departure_planned_time,departure_change_time,arrival_is_canceled,departure_is_canceled
11876245,08000031,RE,4684,0.0,3924581614387547673,6,2026-07-25 22:59:00,2026-07-25 22:59:00,2026-07-25 22:59:00,2026-07-25 22:59:00,False,False
11877671,08000031,RE,4684,5.0,-6707776539338401670,104,2026-07-25 22:59:00,2026-07-25 23:03:00,2026-07-25 22:59:00,2026-07-25 23:04:00,False,False
4569451,08000032,RB,58019,0.0,-748086963037885863,21,2026-07-10 09:37:00,2026-07-10 09:37:00,2026-07-10 09:38:00,2026-07-10 09:38:00,False,False
4569666,08000032,RB,58019,0.0,-8204721622065213175,119,2026-07-10 09:37:00,2026-07-10 09:37:00,2026-07-10 09:38:00,2026-07-10 09:38:00,False,False
5903235,08000032,RB,58019,0.0,-8204721622065213175,119,2026-07-13 09:37:00,2026-07-13 09:37:00,2026-07-13 09:38:00,2026-07-13 09:38:00,True,True
5903450,08000032,RB,58019,0.0,-748086963037885863,21,2026-07-13 09:37:00,2026-07-13 09:37:00,2026-07-13 09:38:00,2026-07-13 09:38:00,False,False


## 3c Extreme delays (count only)

Nothing is excluded here. Whether to exclude, and at what threshold, is an owner decision after seeing these counts.

In [7]:
delay = df[DELAY_COLUMN]
too_late = delay > EXTREME_DELAY_MIN
too_early = delay < -EXTREME_DELAY_MIN
extreme = too_late | too_early

print(f"Threshold (proposal): |delay| > {EXTREME_DELAY_MIN} min")
print("Extreme rows:", share(int(extreme.sum()), int(delay.notna().sum())))
print(f"  Late  (above +{EXTREME_DELAY_MIN}): {int(too_late.sum()):,}")
print(f"  Early (below -{EXTREME_DELAY_MIN}): {int(too_early.sum()):,}")
print("  Also in a ride-key duplicate group:", share(int((extreme & in_ride_group).sum()), int(extreme.sum())))

print("\nExtreme rows by train type:")
display(df.loc[extreme, "train_type"].value_counts().head(10))

Threshold (proposal): |delay| > 180 min
Extreme rows: 1,372 of 14,276,283 (0.0%)
  Late  (above +180): 1,371
  Early (below -180): 1
  Also in a ride-key duplicate group: 1,207 of 1,372 (88.0%)

Extreme rows by train type:


train_type
ICE    326
Bus    292
RE     114
RB     101
NJ      98
RJ      65
ARV     58
FLX     57
IC      52
AVG     22
Name: count, dtype: int64

## 6 Planned hours per service and station

Hypothesis: a recurring service passes a station at about the same hour every day, so "hour" adds little once service x station is known. Service = train type + train number (the board said train number only; the type is added so that, for example, S 1 and RE 1 are never merged). A train planned at 8:59 on some days and 9:01 on others counts as two hours, so a value of 2 is not always a real schedule change.

In [8]:
planned = df["departure_planned_time"].fillna(df["arrival_planned_time"])
hours = planned.dt.hour.rename("planned_hour")
pair_keys = [df[column] for column in [*SERVICE_KEY, "eva"]]
pairs = hours.groupby(pair_keys, observed=True).agg(["size", "nunique"])

one_hour = pairs["nunique"] == 1
print(f"Service x station pairs: {len(pairs):,}")
print("  With exactly one planned hour (share of pairs):", share(int(one_hour.sum()), len(pairs)))
print("  Stops in those pairs (share of stops):         ",
      share(int(pairs.loc[one_hour, "size"].sum()), int(pairs["size"].sum())))

print("\nPairs by number of distinct planned hours:")
display(pairs["nunique"].value_counts().sort_index().head(10))

Service x station pairs: 851,617
  With exactly one planned hour (share of pairs): 828,371 of 851,617 (97.3%)
  Stops in those pairs (share of stops):          13,910,115 of 14,276,283 (97.4%)

Pairs by number of distinct planned hours:


nunique
1     828371
2      20501
3       1446
4        430
5        302
6        188
7        141
8        106
9         56
10        43
Name: count, dtype: int64

## 3f Cancellations vs planned departure

Rows with no planned departure leave P(late). If cancelled stops often have no planned departure, removing those rows from a shared table would also remove most cancellations, and P(cancelled) would need its own base table. This cell checks how much the two overlap.

In [9]:
cancelled = df["arrival_is_canceled"] | df["departure_is_canceled"]
n_cancelled = int(cancelled.sum())
print("Cancelled stops (either flag):", share(n_cancelled, len(df)))
print("  ...with no planned departure:", share(int((cancelled & no_departure).sum()), n_cancelled))

print("\nEither flag vs no planned departure:")
print(pd.crosstab(no_departure.rename("no planned departure"),
                  cancelled.rename("cancelled (either flag)"), margins=True))

print("\nDeparture flag only vs no planned departure:")
print(pd.crosstab(no_departure.rename("no planned departure"),
                  df["departure_is_canceled"].rename("departure cancelled"), margins=True))

Cancelled stops (either flag): 651,697 of 14,768,104 (4.4%)
  ...with no planned departure: 59,349 of 651,697 (9.1%)

Either flag vs no planned departure:
cancelled (either flag)     False    True       All
no planned departure                               
False                    12557421  592348  13149769
True                      1558986   59349   1618335
All                      14116407  651697  14768104

Departure flag only vs no planned departure:
departure cancelled      False    True       All
no planned departure                            
False                 12580489  569280  13149769
True                   1614283    4052   1618335
All                   14194772  573332  14768104


## Not in this notebook

3d (buses) is no longer needed: the owner decided to drop train type Bus and keep SBH. `month_path` is still copied from notebook 01; a later refactor imports it from `src/dbdelay`.